In [90]:
import re
from datetime import datetime
import csv

In [80]:
class BankAccount:
    
    init_num = 1000
    _OWNER_RE = re.compile(r"[А-ЯЁA-Z][а-яёa-z]* [А-ЯЁA-Z][а-яёa-z]*")
    account_type = None
    _VALID_OPERATIONS = ("deposit", "withdraw", "transfer_in", "transfer_out", "interest")
    
    def __init__(self, owner):
        if not isinstance(owner, str) or not self._OWNER_RE.fullmatch(owner):
            raise ValueError(f"Некорректный владелец: {owner!r}.")
        self.owner = owner
        self.account_id = f"ACC-{BankAccount.init_num}"
        BankAccount.init_num += 1
        self.balance = 0.0
        self.history = []
        self.created_at = datetime.now()

    def _updHist(self, operation, amount, status):
        self.history.append({"ts": datetime.now(), "operationType": operation, "amount": amount, "balance": self.balance, "status": status})

    def deposit(self, amount):
        if not isinstance(amount, (int, float)) or isinstance(amount, bool) or amount <= 0:
            self._updHist("deposit", amount, status=False)
            return False
        self.balance += float(amount)
        self._updHist("deposit", float(amount), status=True)
        return True

    def withdraw(self, amount):
        if not isinstance(amount, (int, float)) or isinstance(amount, bool) or amount <= 0:
            self._updHist("withdraw", amount, status=False)
            return False
        if amount > self.balance:
            self._updHist("withdraw", float(amount), status=False)
            return False
        self.balance -= float(amount)
        self._updHist("withdraw", float(amount), status=True)
        return True

    def operatins_over(self, amount):
        if not isinstance(amount, (int, float)) or isinstance(amount, bool) or amount <= 0:
            return []
        return [rec for rec in self.history
                if isinstance(rec["amount"], (int, float)) 
                and not isinstance(rec["amount"], bool) 
                and rec["amount"] > amount]

    def import_history(self, records):
        if not isinstance(records, list):
            return 0, ["ожидался список записей истории"]
        imported, errors = 0, []
        for i, rec in enumerace(records):
            record, reason = self._validate_history_record(rec)
            if reason:
                errors.append(f"запись {i}: {reason}")
            else:
                self.history.append(record)
                imported += 1
        if imported:
            slef.history.sort(key=lambda r: r["ts"])
        return imported, errors

    def _validate_history_record(self, rec):
        if not isinstance(rec, dict):
            return None, "Запись должна быть словарем"
        missing = {"ts", "operationType", "amount", "balance", "status"} - rec.keys()
        if missing:
            return None, f"нет полей {', '.join(sorted(missing))}"
        ts = rec["ts"]
        if isinstance(ts, str):
            try:
                ts = datetime.fromisoformat(ts)
            except ValueError:
                return None, f"некорректная дата: {ts!r}"
        elif not isinstance(ts, datetime):
            return None, f"некорректная дата: {ts!r}"
        operation = rec["operationType"]
        if not isinstance(operation, str) or operation not in self._VALID_OPERATIONS:
            return None, f"неизвестный тип операции: {operation!r}"
        if operation == "interest" and self.account_type != "savings":
            return None, "операция возможна только для сберегательного счета"
        amount = rec["amount"]
        if not isinstance(amount, (int, float)) or isinstance(amount, bool) or amount <= 0:
            return None, f"сумма операции должна быть числом больше 0: {amount!r}"
        balance = rec["balance"]
        if not isinstance(balance, (int, float)) or isinstance(balance, bool) or amount <= 0:
            return None, f"баланс должен быть числом больше 0: {balance!r}"
        status = rec["status"]
        if not isinstance(status, bool):
            return None, f"некорректный статус: {status!r}"
        return ({"ts": ts, "operationType": operation, "amount": float(amount), "balance": float(balance), "status": status}, None)

    def __repr__(self):
        return (f"BankAccount(id={self.account_id!r}, owner={self.owner!r}, "
                f"balance={self.balance})")

In [81]:
class CheckingAccount(BankAccount):
    
    account_type = "checking"

In [82]:
class SavingsAccount(BankAccount):
    
    account_type = "savings"
    _MAX_WITHDRAW_SHARE = 0.5

    def __init__(self, owner, interest_rate):
        super().__init__(owner)
        if not isinstance(interest_rate, (int, float)) or isinstance(interest_rate, bool) or interest_rate < 0:
            raise ValueError(f"Некорректная процентная ставка: {interest_rate!r}")
        self.interest_rate = float(interest_rate)

    def _can_debit(self, amount):
        return (super()._can_debit(amount) and amount <= self.balance * self._MAX_WITHDRAW_SHARE)

    def accure_interest(self):
        interest = round(self.balance * self.interest_rate / 100, 2)
        self.balance += interest
        self.updHist("interest", interest, status=True)
        return interest

    def withdraw(self, amount):
        if (isinstance(amount, (int, float)) and not isinstance(amount, bool) and 0 < amount <= self.balance and amount > self.balance * self._MAX_WITHDRAW_SHARE):
            self._updHist("withdraw", float(amount), status=False)
            return False
        return super().withdraw(amount)

In [83]:
def transfer(sender, recipient, amount):
    if (not isinstance(sender, BankAccount) or not isinstance(recipient, BankAccount) or sender is recipient):
        return False
    if not isinstance(amount, (int, float)) or isinstance(amount, bool) or amount <= 0:
        return False
    if sender.account_type == "savings" and not sender._can_debit(amount):
        sender.UpdHist("transfer_out", float(amount), status=False)
        return False
    sender.balance -= float(amount)
    recipient.balance += float(amount)
    sender._updHist("transfer_out", float(amount), status=True)
    recipient._updHist("transfer_out", float(amount), status=True)
    return True

In [84]:
a = CheckingAccount("Иван Иванов")
b = SavingsAccount("Вася Пупкин", 10)

In [85]:
a.deposit(1000)

True

In [86]:
a.deposit("scdc")

False

In [87]:
b.deposit(500.5)

True

In [88]:
transfer(a, b, 50)

True

In [89]:
b.withdraw(10)

True

In [92]:
result = []
with open('transactions_dirty.csv', mode="r", encoding="utf-8") as file:
    csv_reader = csv.DictReader(file)
    for row in csv_reader:
        result.append(dict(row))

In [93]:
 # поскольку работа выполнялась последовательно, а я не сразу открыл данные, то я не знал их формата, а тут уже дедлайн

[{'account_number': 'ACC-100001',
  'account_type': 'checking',
  'date': '2025-09-27 22:17:26',
  'operation': 'deposit',
  'amount': '921.0',
  'balance_after': '2121.0',
  'status': 'success'},
 {'account_number': 'ACC-100001',
  'account_type': 'checking',
  'date': '2025-09-27 22:17:26',
  'operation': 'deposit',
  'amount': '607.0',
  'balance_after': '2728.0',
  'status': 'success'},
 {'account_number': 'ACC-100001',
  'account_type': 'checking',
  'date': '2025-09-28 22:17:26',
  'operation': 'deposit',
  'amount': '488.0',
  'balance_after': '3216.0',
  'status': 'success'},
 {'account_number': 'ACC-100001',
  'account_type': 'checking',
  'date': '28/09/2025 22:17',
  'operation': 'deposit',
  'amount': '129.0',
  'balance_after': '3345.0',
  'status': 'success'},
 {'account_number': 'ACC-100001',
  'account_type': 'checking',
  'date': '2025-09-29 22:17:26',
  'operation': 'deposit',
  'amount': '880.0',
  'balance_after': '4225.0',
  'status': 'success'},
 {'account_number'